# Task 2, Part B1: Dense and Sliding-Window (W=64) Baselines on MQAR

Fixed protocol for every model (`src/mqar_train.py`): AdamW, peak LR 8e-4, weight decay 0.01, batch 32,
4,500 updates, 200 warmup updates then linear decay to 15% of peak, global-norm clipping 1.0, dev evaluation at
updates 500/1000/1500/2000/3000/4500, seeds 6304/6305/6306. Loss: cross-entropy over the 64 value IDs at the K
marked query positions only (query-position classification, not next-token LM).

Runs: dense K in {4, 8, 16} (mixed regime), dense K=16 near-only control; window W=64 K in {4, 8, 16} (near-only)
and K=16 (mixed). Every model is evaluated on all 9 frozen test cells (K x {near, far, mixed}, 1,024 sequences each).
Isolation control: the trained dense model with the W=64 mask imposed at test time only.

**Architectural access limit:** two W=64 layers can move information at most 2(W-1) = 126 positions, so every
far-regime query (distance >= 128) is outside the window model's receptive field by construction.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task2_attention_kv")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task2", need_public=False, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import save_json, load_json
from shared.src.plotting import save_figure
from task2_attention_kv.src import mqar_data as D
from task2_attention_kv.src.mqar_train import PROTOCOL, train_run, test_eval, run_name
from task2_attention_kv.src.mqar_model import MQARModel, n_params, attended_pairs, state_bytes, receptive_field

OUT = ctx.results / "mqar"
CACHE = ctx.checkpoints / "mqar_cache"
gen_stats = D.verify_generator(64)
save_json(OUT / "generator_checks.json", gen_stats)
dev, test = D.load_frozen("development"), D.load_frozen("test")
print({f"K{k}_{r}": v["tokens"].shape[0] for (k, r), v in test.items()})
pd.DataFrame(gen_stats).T

In [ ]:
RUNS = [("dense", "mixed", 4), ("dense", "mixed", 8), ("dense", "mixed", 16), ("dense", "near", 16),
        ("window", "near", 4), ("window", "near", 8), ("window", "near", 16), ("window", "mixed", 16)]
SEEDS = PROTOCOL["seeds"]
print({k: n_params(MQARModel(k)) for k in ("dense", "window")})

In [ ]:
for seed in SEEDS:
    for kind, regime, k in RUNS:
        h = train_run(kind, k, regime, seed, dev, OUT, CACHE)
        print(f"{h['name']}: {h['train_time_s']:.0f}s final dev {h['dev'][-1]}")

In [ ]:
results = []
for seed in SEEDS:
    for kind, regime, k in RUNS:
        results.append(test_eval(OUT, test, kind, regime, k, seed))
    # test-time-only W=64 mask on the dense models (isolation control)
    results.append(test_eval(OUT, test, "dense", "mixed", 16, seed, eval_kind="window"))
    results.append(test_eval(OUT, test, "dense", "near", 16, seed, eval_kind="window"))

def flatten(results):
    rows = []
    for r in results:
        for cell, v in r["cells"].items():
            k, reg = cell.split("_")
            rows.append(dict(model=f"{r['kind']}->{r['eval_kind']}" if r["eval_kind"] != r["kind"] else r["kind"],
                             train=f"{r['train_regime']} K{r['train_K']}", seed=r["seed"], test_K=int(k[1:]), test_regime=reg,
                             **{kk: vv for kk, vv in v.items()}))
    return pd.DataFrame(rows)
df = flatten(results)
df.to_csv(OUT / "B1_test_cells.csv", index=False)

## Learning curves (mean +- std over seeds)

In [ ]:
def plot_curves(runs, fname):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for kind, regime, k in runs:
        H = [load_json(OUT / "histories" / f"{run_name(kind, regime, k, s)}.json") for s in SEEDS]
        steps = [p["step"] for p in H[0]["train_loss"]]
        L = np.array([[p["loss"] for p in h["train_loss"]] for h in H])
        axes[0].plot(steps, L.mean(0), label=f"{kind} {regime} K{k}")
        axes[0].fill_between(steps, L.mean(0) - L.std(0), L.mean(0) + L.std(0), alpha=0.2)
        dsteps = [p["step"] for p in H[0]["dev"]]
        key = f"K{k}_{regime}"
        A = np.array([[p[key] for p in h["dev"]] for h in H])
        axes[1].errorbar(dsteps, A.mean(0), yerr=A.std(0), marker="o", capsize=3, label=f"{kind} {regime} K{k}")
    axes[0].set_yscale("log"); axes[0].set_xlabel("update"); axes[0].set_ylabel("train CE (query positions)")
    axes[1].set_xlabel("update"); axes[1].set_ylabel("dev accuracy (training K & regime)"); axes[1].axhline(1 / 64, ls=":", c="k")
    axes[1].legend(fontsize=8)
    save_figure(ctx.figures, fname)
plot_curves(RUNS, "mqar_B1_learning_curves")

## Near / far / mixed accuracy by load K (mean +- std over 3 seeds)

In [ ]:
summ = df.groupby(["model", "train", "test_regime", "test_K"])["acc"].agg(["mean", "std"]).round(3)
summ.unstack(["test_regime"]).unstack("test_K")

In [ ]:
same_K = df[df.apply(lambda r: f"K{r.test_K}" in r.train, axis=1)]
tab = same_K.groupby(["model", "train", "test_regime"])["acc"].agg(["mean", "std"]).unstack("test_regime").round(3)
tab.to_csv(OUT / "B1_near_far_mixed_at_training_K.csv")
tab

## Train-time vs test-time window mask (isolation control)

In [ ]:
ctrl = df[(df.test_K == 16) & df.model.isin(["dense", "dense->window", "window"]) & df.train.isin(["mixed K16", "near K16"])]
ctrl.groupby(["model", "train", "test_regime"])[["acc", "acc_d0_32", "acc_d32_64", "acc_d64_128", "acc_d128_256"]].mean().round(3)

## Theoretical cost at T = 256 (per layer) and receptive field

In [ ]:
pd.DataFrame([dict(kind=k, **attended_pairs(k, 256), **state_bytes(k, 256), receptive_field=receptive_field(k))
              for k in ("dense", "window")]).set_index("kind")